In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
import joblib
import os

In [ ]:

# Load data dari file JSON
with open('data/clean_recipes_5000.json', 'r', encoding='utf-8') as f:
    data = json.load(f)


if isinstance(data, list):
    df = pd.DataFrame(data)
else:
    df = pd.DataFrame([data])

df.head()

In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df.describe()

## 2. Praproses Data

Tahapan praproses data meliputi:
- Menghapus duplikasi
- Menangani missing value
- Encoding fitur kategorikal
- Normalisasi fitur numerik

Contoh kode:

In [ ]:
print('Sebelum hapus duplikasi:', len(df))
df = df.drop_duplicates()
print('Setelah hapus duplikasi:', len(df))

# Tangani missing value 
df = df.dropna()

df.head()

In [ ]:
# kolom yang akan dipakai sebagai fitur
fitur_kolom = ['Total Ingredients', 'Total Steps', 'Loves', 'Category']
target_kolom = 'Quality Score'

# siapkan X dan y
X = df[fitur_kolom].copy()
y = df[target_kolom].copy()

print("Bentuk X:", X.shape)
print("Bentuk y:", y.shape)

In [ ]:
# Pilih fitur dan target
fitur = ['Total Ingredients', 'Total Steps', 'Loves', 'Category']
X = df[fitur].copy()
y = df['Quality Score']

# Encode hanya kolom Category
le = LabelEncoder()
X['Category'] = le.fit_transform(X['Category'].astype(str))

# Split data (sebelum scaling)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Scaling fitur numerik (fit hanya dari data train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# SPLIT DATA: TRAIN, VALIDATION, TEST UNTUK MLP
from sklearn.model_selection import train_test_split

# Split awal: train+val dan test (85%:15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Split train dan validation dari X_temp (train:val = 70:15 dari total data)
val_size = 0.15 / 0.85  # proporsi validation dari X_temp
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size, random_state=42)

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

In [ ]:
# Inisialisasi scaler
scaler = StandardScaler()

# Fit hanya pada data train, lalu transform train, val, test
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling selesai.")
print("Contoh nilai pertama train setelah scaling:", X_train_scaled[0])

In [ ]:

# Asumsi X_train_scaled sudah ada
input_shape = X_train_scaled.shape[1]

inputs = tf.keras.Input(shape=(input_shape,), name='input_fitur')

# Hidden layer 1: 128 neuron
x = layers.Dense(128, activation='relu', name='hidden1')(inputs)
x = layers.Dropout(0.3, name='dropout1')(x)  

# Hidden layer 2: 64 neuron (bisa juga 32)
x = layers.Dense(64, activation='relu', name='hidden2')(x)
x = layers.Dropout(0.2, name='dropout2')(x)

# Output layer (regresi untuk Quality Score)
outputs = layers.Dense(1, activation='linear', name='output')(x)

# Model
model = Model(inputs=inputs, outputs=outputs, name='SayurKita_MLP_Quality')
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()